# 01: Building a Micrograd Autograd Engine

## Learning objectives
- Understand what a derivative is and how to approximate it numerically.
- Build a tiny scalar-valued autograd engine from scratch with the `Value` class.
- See how a computation graph records operations so gradients can flow backward.
- Learn why reverse-mode autodiff needs a **topological sort** to apply the chain rule in the right order.
- Visualize computation graphs with `draw_dot`.

## Why this matters
Every modern deep-learning framework (PyTorch, JAX, TensorFlow) is, at its core, an automatic-differentiation engine. By building micrograd yourself, you strip away the magic and see exactly how gradients are computed, stored, and propagated. That intuition is what makes debugging neural nets possible.

## Prerequisites
- Basic Python: functions, classes, lambdas, and dictionaries.
- High-school calculus: derivative, partial derivative, and the chain rule.
- Familiarity with `numpy`, `matplotlib`, and Jupyter notebooks.


In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from nnzero import Value, draw_dot, trace
from nnzero.utils import build_vocab, build_dataset, split_dataset, load_names, set_seed
%matplotlib inline

### Why start with numerical derivatives?
Before we build an automatic system, we approximate derivatives by hand with a tiny perturbation `h`. This gives us an independent way to check that our autograd engine is correct later.

In [ ]:
def f(x):
  return 3*x**2 - 4*x + 5

In [ ]:
f(3.0)

In [ ]:
xs = np.arange(-5, 5, 0.25)
ys = f(xs)
plt.plot(xs, ys)

In [ ]:
h = 0.000001
x = 2/3
(f(x + h) - f(x))/h

In [ ]:
# les get more complex
a = 2.0
b = -3.0
c = 10.0
d = a*b + c
print(d)

In [ ]:
h = 0.0001

# inputs
a = 2.0
b = -3.0
c = 10.0

d1 = a*b + c
c += h
d2 = a*b + c

print('d1', d1)
print('d2', d2)
print('slope', (d2 - d1)/h)


### Why build a tiny autograd engine?
Instead of computing every partial derivative by hand, we wrap each number in a `Value` object that remembers how it was produced. The object stores:
- `data`: the scalar value,
- `grad`: the gradient of the final output with respect to this value,
- `_prev`: parent values,
- `_op`: the operator that created it,
- `_backward`: a closure that applies the local derivative.

This is exactly what PyTorch tensors do behind the scenes.

In [ ]:
class Value:
  
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data})"
  
  def __add__(self, other):
    out = Value(self.data + other.data, (self, other), '+')
    
    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward
    
    return out

  def __mul__(self, other):
    out = Value(self.data * other.data, (self, other), '*')
    
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
      
    return out
  
  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self, ), 'tanh')
    
    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward
    
    return out
  
  def backward(self):
    
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)
    
    self.grad = 1.0
    for node in reversed(topo):
      node._backward()


a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
e = a*b; e.label = 'e'
d = e + c; d.label = 'd'
f = Value(-2.0, label='f')
L = d * f; L.label = 'L'
L

### Try it yourself: add an `exp()` operation to `Value`

The manual `Value` class above supports `+`, `*`, and `tanh`. Add an `exp()` method so that `x.exp()` returns `e^x` and backpropagates correctly.

*Hint:* the local derivative of `exp(x)` is `exp(x)` itself.

In [ ]:
# Your code here
x = Value(2.0, label='x')
# y = x.exp()
# y.backward()
# print(x.grad, y.data)  # should be equal


## Solution (try the exercise before peeking)

In [ ]:
def exp(self):
    out = Value(math.exp(self.data), (self,), 'exp')
    def _backward():
        self.grad += out.data * out.grad
    out._backward = _backward
    return out

Value.exp = exp

x = Value(2.0, label='x')
y = x.exp(); y.label = 'y'
y.backward()
print('x.grad =', x.grad)
print('y.data =', y.data)
print('match?', math.isclose(x.grad, y.data))


### Why visualize the graph?
A picture of the computation graph makes it obvious which values depend on which. The helpers `trace` and `draw_dot` are already implemented in `nnzero`, so we imported them at the top instead of re-writing them here.

In [ ]:
draw_dot(L)

### Why nudge inputs in the direction of the gradient?
The gradient tells us how to change an input to increase the output. If we want to *decrease* a loss, we move in the opposite direction; the size of the step is the learning rate. This single idea is gradient descent.

In [ ]:
a.data += 0.01 * a.grad
b.data += 0.01 * b.grad
c.data += 0.01 * c.grad
f.data += 0.01 * f.grad

e = a * b
d = e + c
L = d * f

print(L.data)


### Why double-check with finite differences?
Our autograd code is easy to get wrong. Finite differences give a trustworthy reference: if `h` is tiny,

```
df/dx ≈ (f(x + h) - f(x)) / h
```

When the analytic gradient matches this estimate, we can be confident the engine is correct.

In [ ]:
def lol():
  
  h = 0.001
  
  a = Value(2.0, label='a')
  b = Value(-3.0, label='b')
  c = Value(10.0, label='c')
  e = a*b; e.label = 'e'
  d = e + c; d.label = 'd'
  f = Value(-2.0, label='f')
  L = d * f; L.label = 'L'
  L1 = L.data
  
  a = Value(2.0, label='a')
  b = Value(-3.0, label='b')
  b.data += h
  c = Value(10.0, label='c')
  e = a*b; e.label = 'e'
  d = e + c; d.label = 'd'
  f = Value(-2.0, label='f')
  L = d * f; L.label = 'L'
  L2 = L.data
  
  print((L2 - L1)/h)
  
lol()

### Why use `tanh` as an activation?
`tanh` squashes any real number into `(-1, 1)` and is smooth everywhere. Its derivative is particularly simple:

```
d/dx tanh(x) = 1 - tanh(x)^2
```

That simplicity is why we use it in this first neuron example.

In [ ]:
plt.plot(np.arange(-5,5,0.2), np.tanh(np.arange(-5,5,0.2))); plt.grid();

### Why hand-build a single neuron?
A neuron is just a weighted sum followed by a non-linearity. Building it manually makes the transition to `torch.nn.Linear` obvious: the math is identical; only the plumbing changes.

In [ ]:
# inputs x1,x2
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
# weights w1,w2
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
# bias of the neuron
b = Value(6.8813735870195432, label='b')
# x1*w1 + x2*w2 + b
x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'

In [ ]:
draw_dot(o)

### Why call `.backward()` on the output?
Calling `o.backward()` means: "I want the gradient of `o` with respect to every value that led to it." The engine then walks the graph backward, applying the chain rule at every step.

In [ ]:
o.backward()

### Why topological sort for backprop?
The chain rule requires that we compute a node's gradient *after* all of its children have been processed. A topological sort guarantees we visit nodes in exactly that order. The next few cells show the same process explicitly, step by step.

In [ ]:
topo = []
visited = set()
def build_topo(v):
  if v not in visited:
    visited.add(v)
    for child in v._prev:
      build_topo(child)
    topo.append(v)
build_topo(o)
topo

In [ ]:
o.grad = 1.0

In [ ]:
o._backward()

In [ ]:
n._backward()

In [ ]:
b._backward()

In [ ]:
x1w1x2w2._backward()

In [ ]:
x2w2._backward()
x1w1._backward()

In [ ]:
x1.grad = w1.data * x1w1.grad
w1.grad = x1.data * x1w1.grad

In [ ]:
x2.grad = w2.data * x2w2.grad
w2.grad = x2.data * x2w2.grad

In [ ]:
x1w1.grad = 0.5
x2w2.grad = 0.5

In [ ]:
x1w1x2w2.grad = 0.5
b.grad = 0.5

In [ ]:
n.grad = 0.5

In [ ]:
o.grad = 1.0

### Try it yourself: verify a gradient numerically

For the neuron example above, approximate `do/dw1` using a small perturbation `h` on `w1.data`. Compare your finite-difference estimate to the analytic gradient produced by `o.backward()`.

In [ ]:
# Your code here
h = 0.001
# ...


## Solution (try the exercise before peeking)

In [ ]:
h = 0.001

# Rebuild the exact same forward pass
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
b = Value(6.8813735870195432, label='b')
x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'
o1 = o.data

# Perturb w1
w1.data += h
x1w1 = x1*w1
x1w1x2w2 = x1w1 + x2w2
n = x1w1x2w2 + b
o2 = n.tanh().data

numeric = (o2 - o1) / h
print('numeric  ', numeric)

# Analytic gradient via backprop
x1 = Value(2.0); x2 = Value(0.0)
w1 = Value(-3.0); w2 = Value(1.0)
b = Value(6.8813735870195432)
o = ((x1*w1 + x2*w2) + b).tanh()
o.backward()
print('analytic ', w1.grad)


### Why is the tanh local derivative `1 - tanh(x)^2`?
This follows directly from calculus. Because `o = tanh(n)`, the chain rule gives:

```
do/dn = (1 - tanh(n)^2) * do/do
      = 1 - o.data**2
```

The next cell computes exactly that expression.

In [ ]:
1 - o.data**2

In [ ]:
# o = tanh(n)
# do/dn = 1 - o**2

### Why do gradients accumulate with `+=`?
A single value can be used more than once (e.g., `b = a + a`). Each use contributes a partial derivative, and the total derivative is the **sum** of those contributions. Using `+=` in `_backward` handles this automatically.

In [ ]:
a = Value(3.0, label='a')
b = a + a   ; b.label = 'b'
b.backward()
draw_dot(b)

### Why must reused values be separate graph nodes?
Even if the same Python object feeds into two branches, the graph treats each edge independently. If `_backward` overwrote instead of accumulating, shared values would silently drop gradient contributions.

In [ ]:
a = Value(-2.0, label='a')
b = Value(3.0, label='b')
d = a * b    ; d.label = 'd'
e = a + b    ; e.label = 'e'
f = d * e    ; f.label = 'f'

f.backward()

draw_dot(f)

### Looking ahead: why Kaiming init, BatchNorm, and cross-entropy?
This notebook built the autograd *engine*. As we stack more layers we will need:
- **Kaiming initialization** — keeps the scale of activations stable through deep networks so gradients don't explode or vanish.
- **Batch normalization** — controls the distribution of layer inputs, allowing higher learning rates and reducing sensitivity to initialization.
- **Cross-entropy loss** — the natural objective for classification; combined with softmax it gives clean gradients and fast convergence.

All three rely on the same chain-rule machinery you just implemented.